In [1]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

In [2]:
augs = pl.scan_csv("augmenter_setups_4_null.csv").filter(
    pl.all_horizontal(pl.col("Damage", "Rate of Fire", "Shield Max").is_not_null()),
    (pl.col("Electric Tempering") <= 0.0) | (pl.col("Electric Tempering").is_null()),
    ~pl.col("").str.contains("Art of Commanding"),
    ~pl.col("").str.contains("Art of Destruction"),
    ~pl.col("").str.contains("Art of Engineering"),
    ~pl.col("").str.contains("Art of Gunnery"),
    ~pl.col("").str.contains("Art of Healing"),
    ~pl.col("").str.contains("Art of Sharpshooting"),
    ~pl.col("").str.contains("Art of Stealth"),
).collect()
# zeros already replaced with nan

# drop class locked augs
# for aug in ['Ultimate Art of Gunnery']:
#     mask = augs.index.str.contains(aug)
#     augs = augs.loc[~mask]


In [3]:
interest = ('Damage', 'Rate of Fire', 'Critical Hit Chance', 'Critical Hit Strength')

In [4]:
args = (pl.col(""), pl.col(*(i for i in interest if i not in ['Electric Tempering'])).rank(descending=True))
if 'Electric Tempering' in interest:
    args += (pl.col("Electric Tempering").rank(descending=False),)

augs_rank = augs.select(*args)

# fill with 2x size
augs_rank = augs_rank.select(
    pl.all().replace({None: 21e6 * 2})
)

In [5]:
augs_rank = augs_rank.with_columns(
    rank=pl.sum_horizontal(*interest)
)

In [6]:
augs = augs.with_columns(
    rank=augs_rank['rank']
)

In [7]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
    display(augs.select(pl.col('', 'rank', *interest)).sort('rank', descending=False).head(15))

,rank,Damage,Rate of Fire,Critical Hit Chance,Critical Hit Strength
str,f64,f64,f64,f64,f64
"""('Celestial Nebulae Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Luminous Obliterative Integrity Augmenter')""",909389.0,3.17,2.66,0.41,2.1
"""('Celestial Nebulae Augmenter', 'Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Luminous Obliterative Integrity Augmenter')""",1.08581e6,2.84,3.07,0.37,2.05
"""('Celestial Nebulae Augmenter', 'Total Eclipse Illunis Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Lustrous Mass Despoliation Augmenter')""",1.106587e6,3.04,2.77,0.37,2.05
"""('Celestial Nebulae Augmenter', 'Total Eclipse Illunis Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Adamanturized Rage Augmenter')""",1165729.5,2.99,2.72,0.37,2.05
"""(""Qa'ik Urk'qii Akk'oj"", 'Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Luminous Obliterative Integrity Augmenter')""",1.180559e6,2.89,2.57,0.37,2.65
…,…,…,…,…,…
"""('Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Lustrous Unalloyed Promontory Augmenter')""",1.318817e6,3.34,2.57,0.35,2.05
"""('Celestial Nebulae Augmenter', 'Empyreal Rage Augmenter', 'Total Eclipse Illunis Augmenter', 'Total Eclipse Illunis Augmenter')""",1.326402e6,2.8,2.35,0.4,2.7
"""('Empyreal Rage Augmenter', 'Luminous Intersticial Desistance Augmenter', 'Luminous Obliterative Integrity Augmenter', 'Luminous Obliterative Integrity Augmenter')""",1359015.5,2.63,3.29,0.34,2.4


In [17]:
# augs.filter(
#     augs[''].str.contains("Art of Fast Killing") & augs[''].str.contains("The Emperor's Augmenter") & augs[''].str.contains("Grand Navigator")
# )#.select(pl.col("", "rank", *interest, 'Electric Tempering', 'Transference Power'))

# augs.filter(pl.col("").str.contains("""Art of Fast Killing Augmenter', 'Ultimate Art of Fast Killing Augmenter', "The Emperor's Augmenter", "Grand Nav"""))
# augs.filter(pl.col("").str.contains("""Art of Fast Killing Augmenter', 'Ultimate Art of Fast Killing Augmenter', "Grand Navigator's Augmenter", "The Emperor's Augmenter"""))
augs.filter(
    pl.col("").str.contains(""""Grand Navigator's Augmenter", 'Ultimate Art of Fast Killing Augmenter', 'Ultimate Art of Fast Killing Augmenter', "The Emperor's""")
)

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Docking Speed,Electric Tempering,Energy Charge,Energy Max,Hostility,Inertial Dampening,Multifiring,Projectile Velocity,Radar,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Tractor Power,Tractor Range,Transference Power,Turning,Visibility,Weapon Hold,Weapons Slots,Weight,rank
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""(""Grand Navigator's Augmenter""…",null,0.3,0.8,1.75,null,-0.413246,1.25,0.45,null,null,null,null,null,0.5,1.8,0.3,0.95,0.45,0.8,null,null,null,null,null,null,null,null,null,null,8.853193e6


In [19]:
augs.filter(
    pl.col("Critical Hit Chance") >= 0.4,
    pl.col("Critical Hit Strength") >= 0.8,
    pl.col("Damage") >= 1.8,
    pl.col("Electric Tempering") <= -0.4,
    pl.col("Energy Max") >= 0.45,
    pl.col("Range") >= 0.5,
    pl.col("Rate of Fire") >= 1.8,
    pl.col("Shield Max") >= 1.0,
    pl.col("Capacity") >= 0.0,
).drop(
    "Docking Speed",
    "Hostility",
    "Inertial Dampening",
    "Multifiring",
    "Projectile Velocity",
    "Radar",
    "Tractor Power",
    "Tractor Range",
    "Visibility",
    "Weapon Hold",
    "Weapons Slots"
)

,Capacity,Critical Hit Chance,Critical Hit Strength,Damage,Electric Tempering,Energy Charge,Energy Max,Range,Rate of Fire,Resistance to Damage,Shield Max,Shield Recovery,Speed,Thrust,Tracking,Transference Power,Turning,Weight,rank
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""('Divine Wattage Augmenter', ""…",0.15,0.4,1.6,1.95,-0.558341,0.3,1.0,0.5,1.85,0.35,2.15,0.55,0.13,null,0.75,null,null,null,5.029218e6
"""('Divine Wattage Augmenter', ""…",0.15,0.4,1.0,2.0,-0.47026,0.3,1.0,0.5,1.8,0.3,4.25,null,null,null,0.75,null,null,null,6.319095e6
"""('Divine Wattage Augmenter', '…",0.15,0.4,1.0,1.9,-0.558341,0.3,1.0,0.5,2.35,0.3,2.75,null,0.13,null,0.75,null,null,null,5626861.5
"""('Divine Wattage Augmenter', '…",0.15,0.4,1.0,2.1,-0.521008,0.3,1.0,0.5,2.05,0.3,2.75,null,0.1,0.2,1.1,null,0.4,null,5.287046e6
"""('Divine Wattage Augmenter', '…",0.15,0.4,1.0,2.05,-0.558341,0.3,1.0,0.5,2.0,0.3,2.75,null,null,null,0.75,null,null,null,5609935.5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""('Luminous Intersticial Desist…",0.15,0.48,1.0,2.3,-0.462699,0.25,1.25,1.15,2.1,0.42,2.25,null,null,null,1.32,null,null,2.0,4359771.5
"""('Luminous Intersticial Desist…",0.15,0.4,1.0,2.5,-0.402126,0.4,0.8,1.07,2.0,null,2.15,null,null,null,1.95,null,null,null,4288977.5
"""('Luminous Intersticial Desist…",0.15,0.5,1.0,2.2,-0.462699,0.25,1.25,1.1,2.1,0.05,2.25,null,null,null,1.95,null,null,null,4.623873e6


In [21]:
augs.filter(
    pl.col("Critical Hit Chance") >= 0.4,
    pl.col("Critical Hit Strength") >= 0.8,
    pl.col("Damage") >= 1.8,
    pl.col("Electric Tempering") <= -0.4,
    pl.col("Energy Max") >= 0.45,
    pl.col("Range") >= 0.5,
    pl.col("Rate of Fire") >= 1.8,
    pl.col("Shield Max") >= 1.0,
    pl.col("Capacity") >= 0.0,
).write_excel("sd_subset.xlsx")